# Azure ML & AI Foundry — Assignment

**Deliverables:** GitHub repo + 2-page PDF report  
**Audience:** Fresh-graduate AI engineers (independent work)

Pick **ONE** of the two tracks below and complete it end-to-end:

- **Track A** — Production-grade Azure ML pipeline (classical ML)
- **Track B** — Evaluated GenAI application with AI Foundry

Most code cells are intentionally left blank with `# TODO` comments — you fill them in.  
Library imports and Azure connections are pre-filled to save you time.

### Grading rubric (100 points)

| Points | Category | What we look for |
|---|---|---|
| 30 | Correctness | Does it run end-to-end? |
| 25 | Engineering quality | Clean code, version control, error handling |
| 25 | Evaluation rigor | Meaningful metrics, honest analysis |
| 20 | Report | Clarity, insight, what you'd do next |

# Track B — Evaluated GenAI Application

**Goal:** Build a RAG app over your own knowledge base, then evaluate and red-team it like a real production system.

Skip this track if you picked Track A.

## B.0 Setup (pre-filled — just run it)

In [ ]:
!pip install -q azure-ai-projects==1.0.0 azure-ai-evaluation==1.5.0 azure-ai-inference==1.0.0b9 openai==1.55.0

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.evaluation import (
    evaluate,
    GroundednessEvaluator,
    RelevanceEvaluator,
    CoherenceEvaluator,
    FluencyEvaluator,
    SimilarityEvaluator,
    HateUnfairnessEvaluator,
    ViolenceEvaluator,
)
from azure.identity import DefaultAzureCredential
import json
import os
import time

# Fill from Foundry portal -> your project -> Overview
PROJECT_ENDPOINT = "https://<your-foundry-account>.services.ai.azure.com/api/projects/<your-project>"

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

# Judge LLM used by the quality evaluators
JUDGE = {
    "azure_endpoint":   "https://<your-foundry-account>.openai.azure.com/",
    "api_key":          "<your-key>",
    "azure_deployment": "gpt-4o",
    "api_version":      "2024-10-21",
}

print("Connected to Foundry project.")

## B.1 Pick a domain and gather 10–20 documents

Pick a **domain** that interests you — legal FAQ, medical first-aid, customer support, education, internal HR — your choice.

Gather **10–20 short documents** (plain text or PDF excerpts of under 500 words each). Put them in a Python dictionary `DOCS = {"doc_id": "text", ...}` or load from files.

In [ ]:
# TODO: Define your DOCS dictionary or load files into one.
# TODO: Write a retriever function retrieve(query, k=3) that returns the top-k docs.
# Hint: keyword overlap is fine; you may also use sentence-transformers.

DOCS = {
    # "doc1": "...",
    # "doc2": "...",
}

def retrieve(query, k=3):
    # Your code here
    pass


## B.2 Build the RAG flow

The `ask(query, model_name)` function should:

1. Retrieve top-k docs with your retriever
2. Build a prompt with the context
3. Call the LLM
4. Return a dict with keys: `query`, `response`, `context`, `ground_truth` (leave ground_truth blank for now)

In [ ]:
def ask(query, model_name="gpt-4o-mini"):
    # TODO: Get the OpenAI client:
    #   client = project.inference.get_azure_openai_client(api_version="2024-10-21")
    # TODO: Retrieve context, build messages, call client.chat.completions.create(...)
    # TODO: Return the structured dict described above
    pass

# Smoke test (uncomment when ready)
# print(ask("your test question here"))


## B.3 Hand-author a 20-row evaluation dataset

Your dataset must contain a mix of:

- **10 happy-path** questions — answers should be in your docs
- **5 edge cases** — empty input, very long input, ambiguous phrasing, multi-language
- **5 adversarial** — prompt injection attempts, off-topic asks, role-play tricks

Each row must have `query`, `response`, `context`, `ground_truth`.

In [ ]:
# TODO: Build a list of 20 (query, ground_truth) tuples.
# TODO: For each query, call ask() to populate response + context.
# TODO: Write the result to eval_dataset.jsonl.

QUERIES = [
    # ("question 1", "expected substring or sentence"),
    # ...
]

# Your code here


## B.4 Run all 5 quality + 2 safety evaluators

Required evaluators:

- **Quality (5):** Groundedness, Relevance, Coherence, Fluency, Similarity
- **Safety (2):** HateUnfairness, Violence

In [ ]:
# TODO: Call evaluate(...) with all 7 evaluators on eval_dataset.jsonl.
# TODO: Save the per-row results to eval_results.json.
# TODO: Print the aggregate metrics dictionary.

# Your code here


## B.5 Write ONE custom evaluator

Pick one of:

- **Response length** — flag responses outside 20–300 words
- **Tone match** — match a target tone (formal / friendly) using the judge LLM
- **Language match** — verify the response is in the same language as the question

A custom evaluator is just a callable that takes `**kwargs` and returns a dict of scores.

In [ ]:
class MyCustomEvaluator:
    """TODO: implement your custom evaluator."""

    def __init__(self, **kwargs):
        # Initialize anything you need (LLM client, thresholds, ...)
        pass

    def __call__(self, *, query, response, **kwargs):
        # Return a dict like {"my_score": 0.85, "my_score_reason": "..."}
        # Your code here
        return {}

# TODO: Re-run evaluate() including MyCustomEvaluator and inspect the new column.


## B.6 Red Teaming Agent — find 3 vulnerabilities

Use the Foundry AI Red Teaming Agent to probe your app. Document 3 attacks that succeeded (fully or partially).

In [ ]:
# TODO: Use azure.ai.evaluation.red_team.RedTeam to run a scan.
# TODO: Configure target = your ask() function.
# TODO: Capture the report and save it to red_team_report.json.

# Your code here


**Vulnerabilities you found:**

1. _Attack type and what happened…_
2. _Attack type and what happened…_
3. _Attack type and what happened…_

**For each vulnerability, what would you change in the prompt or system to fix it?**

_Your answers here._

## B.7 Compare two models

Re-run the evaluation with `gpt-4o` as the target model (instead of `gpt-4o-mini`). Build a side-by-side comparison table:

| Model | Avg Groundedness | Avg Relevance | Latency P95 | Tokens per call |

In [ ]:
# TODO: Run the same eval set against both gpt-4o-mini and gpt-4o.
# TODO: Record latency and token counts during inference.
# TODO: Build the comparison DataFrame.

# Your code here


## B.8 Reflection — where does your app shine and break?

Answer in 4–6 sentences below:

- What kinds of queries does it handle well?
- What kinds break it?
- Which evaluator caught the most real problems?
- If you had another week, what would you fix first?

**Your reflection:**

_Write your reflection here._

# Deliverables Checklist

Before you submit, verify:

- [ ] GitHub repo is public or shared with the instructor
- [ ] README explains how to set up and run your notebook
- [ ] Screenshots from the Foundry portal or Azure ML Studio are in `/screenshots`
- [ ] Your 2-page reflection PDF is in the repo root as `REPORT.pdf`
- [ ] All Azure resources you created have been **cleaned up** (no orphan endpoints!)

**Submission deadline:** one week from today.

Good luck — build something you would be proud to show in an interview.